# Piyu Teacher — free fine-tune (LoRA) on Google Colab / Kaggle

1. Runtime → Change runtime type → **T4 GPU** (free).
2. Upload `dataset.jsonl` (made with `node finetune/make_dataset.js state.json > dataset.jsonl`) using the Files panel on the left.
3. Run all cells (about 20–40 minutes for ~1000 examples).
4. Download `piyu-teacher.Q4_K_M.gguf`, then follow `finetune/README.md` to put it on the VPS.

Everything here is free and open source. If a library API changed, the error message usually tells the new name.

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch
BASE = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"     # use "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit" if the GPU runs out of memory
model, tokenizer = FastLanguageModel.from_pretrained(model_name=BASE, max_seq_length=2048, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=32, lora_dropout=0, target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"], use_gradient_checkpointing="unsloth")

In [ ]:
from datasets import load_dataset
ds = load_dataset("json", data_files="dataset.jsonl", split="train")
def fmt(ex): return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False)}
ds = ds.map(fmt)
print(len(ds)); print(ds[0]["text"][:600])

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds, args=SFTConfig(
    dataset_text_field="text", max_seq_length=2048, per_device_train_batch_size=2, gradient_accumulation_steps=4,
    num_train_epochs=2, learning_rate=2e-4, warmup_steps=10, logging_steps=10, optim="adamw_8bit", output_dir="out", report_to="none", fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported()))
trainer.train()

In [ ]:
# quick test: ask it something grounded
FastLanguageModel.for_inference(model)
msgs=[{"role":"system","content":"You are Piyu, a calm Indian teacher. Answer only from the excerpts."},{"role":"user","content":"DOCUMENT EXCERPTS:\n[1] Photosynthesis is the process by which green plants prepare their own food using sunlight, carbon dioxide and water.\n\nQUESTION: Photosynthesis kya hai?"}]
ids=tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
print(tokenizer.decode(model.generate(input_ids=ids, max_new_tokens=120, temperature=0.2)[0][ids.shape[1]:], skip_special_tokens=True))

In [ ]:
# export a small file for Ollama (4-bit GGUF) and download it
model.save_pretrained_gguf("piyu-teacher", tokenizer, quantization_method="q4_k_m")
import glob; print(glob.glob("piyu-teacher*/*.gguf") + glob.glob("*.gguf"))

In [ ]:
from google.colab import files
import glob
for f in glob.glob("piyu-teacher*/*.gguf") + glob.glob("*.gguf"): files.download(f)